In [ ]:
# Install dependencies
!pip install -q timm pytorch-msssim lpips scikit-image thop gdown transformers accelerate

import os
import shutil

# 1. Clone repository
!git clone https://github.com/sumire25/RSHazeNet.git /kaggle/working/RSHazeNet
%cd /kaggle/working/RSHazeNet

# 2. Setup RICE paths
RICE1_CLOUD = '/kaggle/input/datasets/shubhank001/rice-remote-sensing-images-for-cloud-removal/RICE/RICE1/cloud'
RICE2_CLOUD = '/kaggle/input/datasets/shubhank001/rice-remote-sensing-images-for-cloud-removal/RICE/RICE2/cloud'

# Kaggle sometimes drops the username prefix when mounting datasets. 
# These fallbacks ensure the paths are found regardless of Kaggle's mount logic.
if not os.path.exists(RICE1_CLOUD):
    RICE1_CLOUD = '/kaggle/input/rice-remote-sensing-images-for-cloud-removal/RICE/RICE1/cloud'
if not os.path.exists(RICE2_CLOUD):
    RICE2_CLOUD = '/kaggle/input/rice-remote-sensing-images-for-cloud-removal/RICE/RICE2/cloud'

CAPTIONS_DIR = '/kaggle/working/captions'
os.makedirs(CAPTIONS_DIR, exist_ok=True)

def caption_path(input_dir):
    key = input_dir.replace('/', '_').strip('_')
    out_dir = os.path.join(CAPTIONS_DIR, key)
    os.makedirs(out_dir, exist_ok=True)
    return os.path.join(out_dir, 'captions.json')

# 3. Initialize VLM and execute loop
import caption
from tqdm import tqdm

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("Initializing Qwen2-VL into VRAM...")
caption_fn, cleanup = caption.build_captioner(
    vlm_model_id="Qwen/Qwen2-VL-2B-Instruct",
    api_key="", api="local", device="cuda", max_side=512
)

def process_directory(input_dir):
    if not os.path.isdir(input_dir):
        print(f"Skipping (Directory not found): {input_dir}")
        return
        
    out_dir = os.path.dirname(caption_path(input_dir))
    os.makedirs(out_dir, exist_ok=True)
    cache_path = os.path.join(out_dir, "captions.json")
    
    cache = caption.load_existing(cache_path)
    images = caption.list_images(input_dir)
    todo = [f for f in images if f not in cache]
    
    print(f'Captioning: {input_dir} | Total Images: {len(images)} | Remaining to process: {len(todo)}')
    
    if not todo: return

    for i, fname in enumerate(tqdm(todo, desc="Processing")):
        ipath = os.path.join(input_dir, fname)
        try:
            text = caption_fn(ipath)
            level = caption.parse_level(text)
            cache[fname] = {"caption": text, "level": level}
        except Exception as e:
            pass
        
        # Save every 20 images to prevent data loss during long runs
        if (i + 1) % 20 == 0 or (i + 1) == len(todo):
            caption.save_cache(cache_path, cache)

# 4. Process the targeted RICE directories
process_directory(RICE1_CLOUD)
process_directory(RICE2_CLOUD)

cleanup()
print("\nRICE1 and RICE2 datasets successfully captioned and saved to /kaggle/working/captions!")